# Tanush — Week 2 ECG-FM in-distribution training

This notebook trains one ECG-FM checkpoint per available source dataset using frozen five-label manifests. It refuses to invent new splits. Week 2 produces only the in-distribution diagonal; Week 3 reuses these checkpoints for cross-dataset evaluation.

In [ ]:
# Safe defaults: verify all data, then run one real model smoke test.
RUN_DATA_CHECKS = True
RUN_MODEL_SMOKE_TEST = True
RUN_FULL_TRAINING = False  # change to True only after the smoke test passes
FULL_DATASETS = ['ptbxl', 'cpsc2018', 'georgia', 'mimic_iv']
print({'data_checks': RUN_DATA_CHECKS, 'model_smoke': RUN_MODEL_SMOKE_TEST, 'full_training': RUN_FULL_TRAINING})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
candidates = [
    Path('/content/drive/MyDrive/LSTS/ecg_benchmark_1/week2_tanush'),
    Path('/content/drive/MyDrive/ecg_benchmark_1/week2_tanush'),
    Path('/content/drive/MyDrive/LSTS/week2_tanush'),
]
WEEK2_DRIVE = next((path for path in candidates if path.exists()), None)
assert WEEK2_DRIVE is not None, f'Could not find week2_tanush. Checked: {candidates}'
WORKSPACE = Path('/content/ecg-modeling-benchmark')
RUN_ROOT = WEEK2_DRIVE / 'runs'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print({'week2_drive': str(WEEK2_DRIVE), 'run_root': str(RUN_ROOT)})

In [ ]:
import os, subprocess, sys
bundle = WEEK2_DRIVE / 'tanush_week2_code.zip'
assert bundle.exists(), f'Missing code bundle: {bundle}'
subprocess.check_call(['unzip', '-q', '-o', str(bundle), '-d', str(WORKSPACE)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'scikit-learn', 'wfdb'])
fairseq_root = Path('/content/fairseq-signals')
if not fairseq_root.exists():
    subprocess.check_call(['git', 'clone', '-q', '--depth', '1', 'https://github.com/Jwoo5/fairseq-signals.git', str(fairseq_root)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(fairseq_root)], env={**os.environ, 'MAX_JOBS': '2'})
sys.path.insert(0, str(WORKSPACE))
print({'workspace_ready': WORKSPACE.exists(), 'fairseq_signals': str(fairseq_root)})

In [ ]:
from huggingface_hub import hf_hub_download
checkpoint_path = Path(hf_hub_download(
    repo_id='wanglab/ecg-fm',
    filename='mimic_iv_ecg_physionet_pretrained.pt',
    local_dir='/content/checkpoints/ecg-fm',
))
print({'checkpoint': str(checkpoint_path), 'size_mb': round(checkpoint_path.stat().st_size / 1e6, 1)})

In [ ]:
import shutil, tarfile

def first_existing(paths):
    found = next((Path(path) for path in paths if Path(path).exists()), None)
    assert found is not None, f'None of these paths exists: {paths}'
    return found

SHARED_PROCESSED = first_existing([
    '/content/drive/MyDrive/LSTS/ecg_benchmark/processed',
    '/content/drive/MyDrive/ecg_benchmark/processed',
])
GEORGIA_DRIVE = first_existing([
    '/content/drive/MyDrive/LSTS/ecg_benchmark_1/processed/georgia',
    '/content/drive/MyDrive/ecg_benchmark_1/processed/georgia',
])
MIMIC_DRIVE = first_existing([
    '/content/drive/MyDrive/LSTS/ecg_benchmark_1/week1_tanush/data/artifacts/mimic_50k_waveforms',
    '/content/drive/MyDrive/ecg_benchmark_1/week1_tanush/data/artifacts/mimic_50k_waveforms',
])
LOCAL_DATA = Path('/content/ecg-data')
LOCAL_DATA.mkdir(exist_ok=True)
print({'shared_processed': str(SHARED_PROCESSED), 'georgia_drive': str(GEORGIA_DRIVE), 'mimic_drive': str(MIMIC_DRIVE)})

In [ ]:
# Copy PTB-XL/CPSC once to local Colab disk; repeated training reads from Drive are too slow.
for dataset in ('ptbxl', 'cpsc2018'):
    destination = LOCAL_DATA / dataset
    if not destination.exists():
        shutil.copytree(SHARED_PROCESSED / dataset, destination)

# Reassemble the split Georgia archive in numeric order and extract locally.
georgia_root = LOCAL_DATA / 'georgia'
if not (georgia_root / 'signals').exists():
    archive = Path('/content/georgia_signals.tar.gz')
    with archive.open('wb') as output:
        for part_number in range(7):
            whole = GEORGIA_DRIVE / f'georgia_signals.tar.gz.part-{part_number:02d}'
            parts = [whole] if whole.exists() else sorted(GEORGIA_DRIVE.glob(f'georgia_signals.tar.gz.part-{part_number:02d}.sub-*'))
            assert parts, f'Missing Georgia archive part {part_number:02d}'
            for part in parts:
                with part.open('rb') as source:
                    shutil.copyfileobj(source, output, length=8 * 1024 * 1024)
    georgia_root.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, 'r:gz') as handle:
        handle.extractall(georgia_root, filter='data')

# Extract MIMIC shards and the v3 replacement overlay.
mimic_parent = LOCAL_DATA / 'mimic_iv'
mimic_root = mimic_parent / 'mimic_50k_waveforms'
if not (mimic_root / 'files').exists():
    mimic_parent.mkdir(parents=True, exist_ok=True)
    for archive in sorted(MIMIC_DRIVE.glob('mimic_50k_waveforms_*-of-050.tar.gz')):
        with tarfile.open(archive, 'r:gz') as handle:
            handle.extractall(mimic_parent, filter='data')
    overlay_dir = MIMIC_DRIVE / 'mimic_50k_v3_replacement_overlay'
    for archive in sorted(overlay_dir.glob('*.tar.gz')):
        with tarfile.open(archive, 'r:gz') as handle:
            handle.extractall(mimic_parent, filter='data')

SIGNAL_ROOTS = {
    'ptbxl': LOCAL_DATA / 'ptbxl',
    'cpsc2018': LOCAL_DATA / 'cpsc2018',
    'georgia': georgia_root,
    'mimic_iv': mimic_root,
}
print({name: str(path) for name, path in SIGNAL_ROOTS.items()})

In [ ]:
MANIFEST_ROOT = WORKSPACE / 'manifests'
def run_pipeline(dataset, extra_args):
    command = [
        sys.executable, str(WORKSPACE / 'src/training/ecg_fm_pipeline.py'),
        '--dataset', dataset,
        '--manifest', str(MANIFEST_ROOT / f'{dataset}_week2.csv'),
        '--signal-root', str(SIGNAL_ROOTS[dataset]),
        '--pretrained-checkpoint', str(checkpoint_path),
        '--output-dir', str(RUN_ROOT / dataset),
        *extra_args,
    ]
    print('Running:', ' '.join(command))
    subprocess.check_call(command, cwd=WORKSPACE)

if RUN_DATA_CHECKS:
    for dataset in FULL_DATASETS:
        run_pipeline(dataset, ['--data-only', '--max-records-per-split', '2'])
    print('DATA CHECKS: PASS for all four available datasets')

In [ ]:
if RUN_MODEL_SMOKE_TEST:
    run_pipeline('georgia', [
        '--smoke-test', '--epochs', '1', '--patience', '1',
        '--max-records-per-split', '512', '--batch-size', '4',
        '--gradient-accumulation-steps', '1',
    ])
    print('ECG-FM MODEL SMOKE TEST: PASS')

In [ ]:
if RUN_FULL_TRAINING:
    for dataset in FULL_DATASETS:
        run_pipeline(dataset, [
            '--epochs', '50', '--patience', '10', '--batch-size', '4',
            '--gradient-accumulation-steps', '8', '--learning-rate', '1e-6',
        ])
else:
    print('Full training is OFF. Turn it on only after reviewing the smoke-test output.')
print('CODE-II remains blocked until its final manifest and signals are uploaded by the assigned teammate.')

In [ ]:
# Consolidate only completed, non-smoke test rows. Missing runs remain explicit.
result_rows = []
for dataset in ['ptbxl', 'cpsc2018', 'georgia', 'mimic_iv', 'code_ii']:
    metrics_path = RUN_ROOT / dataset / 'test_metrics.csv'
    if metrics_path.exists():
        row = __import__('pandas').read_csv(metrics_path).iloc[0].to_dict()
        row['availability'] = row.get('status', 'COMPLETE')
    else:
        row = {'architecture': 'ECG-FM', 'dataset': dataset, 'availability': 'NOT_RUN' if dataset != 'code_ii' else 'BLOCKED_MISSING_DATA'}
    result_rows.append(row)
results = __import__('pandas').DataFrame(result_rows)
results.to_csv(WEEK2_DRIVE / 'ecg_fm_week2_status.csv', index=False)
display(results)